# Notebook 35 — Support-Safe Matched Diagnostics (Auth A)

**Authorization:** Auth A only (Phases 0–3). Auth B / B+ / C are **not** executed.

**Question:** After replacing clipping with support-safe rejection sampling, which calibration phenotypes remain as residual failures on the powered Development bank?

This notebook **reads** artifacts under `results/figure10_8d_7d/artifacts/notebook_35/`. Heavy computation: `conda run -n neurolib python -m sleep_sbi.figure10_notebook35_audit --phase all`.

**Sealed Bank:** `SEALED_NOT_OPENED`. **131k / 1M:** NO-GO.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
N35 = ROOT / "results" / "figure10_8d_7d" / "artifacts" / "notebook_35"
JSON = N35 / "json"
CSV = N35 / "csv"

exe = Path(os.__file__).resolve().parents[1]  # noqa: not used as env proof
assert "neurolib" in os.environ.get("CONDA_DEFAULT_ENV", "") or "neurolib" in str(
    Path(__import__("sys").executable)
).lower(), "Run via conda run -n neurolib / Python (neurolib) kernel"

required = [
    JSON / "protocol_lock.json",
    JSON / "input_artifact_manifest.json",
    JSON / "environment_report.json",
    JSON / "auth_a_decision_gate.json",
    CSV / "phase2_matched_paired_deltas.csv",
    CSV / "residual_failure_registry_primary.csv",
]
missing = [p for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing Auth A artifacts. From S4_sbi/src run:\n"
        "  conda run -n neurolib python -m sleep_sbi.figure10_notebook35_audit --phase all\n"
        + "\n".join(str(p) for p in missing)
    )

lock = json.loads((JSON / "protocol_lock.json").read_text(encoding="utf-8"))
env = json.loads((JSON / "environment_report.json").read_text(encoding="utf-8"))
manifest = json.loads((JSON / "input_artifact_manifest.json").read_text(encoding="utf-8"))
gate = json.loads((JSON / "auth_a_decision_gate.json").read_text(encoding="utf-8"))
print("Artifacts OK:", N35)
print("Auth tier:", gate["auth_tier"], "| Sealed:", gate["sealed_status"])

Artifacts OK: D:\Year3_Mao_Projects\sleep_loop\S4_sbi\results\figure10_8d_7d\artifacts\notebook_35
Auth tier: A | Sealed: SEALED_NOT_OPENED


## 1. Executive scope and inherited evidence

Notebook 34 concluded `A_IMPLEMENTATION_FIX_FIRST` with material clip sensitivity. Auth A completes support-safe rejection sampling and re-evaluates residual phenotype on powered_1024. matched-48 estimates the paired clip vs rejection effect only.

In [2]:
display(Markdown(
    f"| Item | Value |\n|---|---|\n"
    f"| Auth tier | `{gate['auth_tier']}` |\n"
    f"| Stopping decision | `{gate['stopping_decision']}` |\n"
    f"| Reason | {gate['reason']} |\n"
    f"| Sealed | `{gate['sealed_status']}` |\n"
    f"| 131k / 1M | `{gate['pilot_131k']}` / `{gate['scale_1M']}` |\n"
    f"| Retraining | `{gate['retraining_performed']}` |\n"
    f"| Manifest entries | {manifest['n_entries']} |\n"
    f"| Python | `{env['sys_executable']}` |\n"
))
display(Markdown("### Primary residual counts"))
display(pd.DataFrame([gate["primary_counts"]]))

| Item | Value |
|---|---|
| Auth tier | `A` |
| Stopping decision | `AUTH_A_COMPLETE_RESIDUAL_PRESENT — AUTH_B_NOT_EXECUTED` |
| Reason | residual_failure_count=1; Auth B not authorized in this run |
| Sealed | `SEALED_NOT_OPENED` |
| 131k / 1M | `NO-GO` / `NO-GO` |
| Retraining | `False` |
| Manifest entries | 633 |
| Python | `C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe` |


### Primary residual counts

,A_REPAIRED,INCONCLUSIVE,NO_FAILURE,RESIDUAL_FAILURE
0,0,5,0,1


## 2. Frozen protocol and input provenance

In [3]:
display(Markdown(
    f"Protocol lock sha256 file present; auth_tier={lock['auth_tier']}; "
    f"matched48 jobs={lock['evaluation_sets']['matched48']['jobs']}; "
    f"powered jobs={lock['evaluation_sets']['powered1024_residual']['jobs']}; "
    f"8192 control jobs={lock['evaluation_sets']['control8192']['jobs']} (excluded from residual)."
))
man_df = pd.DataFrame(manifest["entries"])[["role", "path", "sha256", "immutable_read_only"]]
display(man_df.head(12))
display(Markdown(f"Showing 12 / {len(man_df)} manifest rows (full table on disk)."))

Protocol lock sha256 file present; auth_tier=A; matched48 jobs=576; powered jobs=12288; 8192 control jobs=96 (excluded from residual).

,role,path,sha256,immutable_read_only
0,notebook_33,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\notebo...,ffc45c38a889098e456ded2d9252193a48c33d5bee8c9b...,True
1,notebook_34,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\notebo...,99a63d1de72223846b7e0a828757baadce22882c87f4b2...,True
2,n33_json:evaluation_protocol_lock.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,cae5a9e4b16a0e1085460daee45fb8945ab1ea247e537e...,True
3,n33_json:matched_geometry_subset_lock.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,48dcd6a9c5a80cb6fa3e77c4332f767ceeb95ec37d5d95...,True
4,n33_json:final_claim_card.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,76714c03b2c4f291a4fbd5727da77c99d1d9fe25fd202d...,True
5,n33_json:provisional_decision_tree.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,5265f0713300475a84fcbfe7b165657a5c7b9972ec0bb4...,True
6,n33_json:phase0_gate.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,d63f19b691a0ace80402244bd5d62becc687b40ca98e8b...,True
7,n33_json:ppc_protocol_lock.json,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,7a48e8f540b5e83d15b1bd00005a582002a4b5242646f9...,True
8,n33_table:attribution_matrix.csv,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,e7ccb4cd3f6407d4cab71048695ad006ed2077179af5dc...,True
9,n33_table:focused_params_mui_mue_tauA.csv,D:\Year3_Mao_Projects\sleep_loop\S4_sbi\result...,a6a704c96fead69dea282375fa3c71ce8a4a886940317c...,True


Showing 12 / 633 manifest rows (full table on disk).

## 3. Matched-48 clipping versus support-safe (paired A effect)

Strictly matched shared archived N34 proposals; continuation used only by rejection branch (conditionally matched).

In [4]:
paired = pd.read_csv(CSV / "phase2_matched_paired_deltas.csv")
focus = paired[(paired.is_focus) & (paired.member_key == "ensemble")].copy()
display(focus[
    [
        "track",
        "parameter",
        "central_cov_90_clipped",
        "central_cov_90_support_safe",
        "delta_central_cov_90",
        "rank_mean_clipped",
        "rank_mean_support_safe",
        "delta_rank_mean",
        "continuation_frac",
        "material_cov",
        "material_rank",
    ]
])
p2 = json.loads((JSON / "phase2_summary.json").read_text(encoding="utf-8"))
display(Markdown(
    f"Phase2 jobs_done={p2['jobs_done']}/{p2['jobs_expected']}; "
    f"continuation_frac_overall={p2['continuation_frac_overall']:.3f}."
))
c8192 = json.loads((JSON / "phase2_control8192_summary.json").read_text(encoding="utf-8"))
display(Markdown(
    f"8192 descriptive control jobs_done={c8192['jobs_done']}/{c8192['jobs_expected']} "
    f"(excluded from residual registry)."
))

,track,parameter,central_cov_90_clipped,central_cov_90_support_safe,delta_central_cov_90,rank_mean_clipped,rank_mean_support_safe,delta_rank_mean,continuation_frac,material_cov,material_rank
4,7d,mue,0.958333,0.875000,-0.083333,0.542032,0.559048,0.017016,1.0,True,False
5,7d,mui,1.000000,1.000000,0.000000,0.413986,0.405783,-0.008203,1.0,False,False
6,7d,tauA,0.895833,0.875000,-0.020833,0.555393,0.571719,0.016326,1.0,False,False
47,8d,mue,0.958333,0.895833,-0.062500,0.537037,0.560876,0.023839,1.0,True,False
48,8d,mui,0.979167,0.958333,-0.020833,0.542641,0.548489,0.005848,1.0,False,False
49,8d,tauA,0.895833,0.875000,-0.020833,0.555028,0.581019,0.025991,1.0,False,False


Phase2 jobs_done=576/576; continuation_frac_overall=1.000.

8192 descriptive control jobs_done=96/96 (excluded from residual registry).

## 4. Residual Failure Registry (powered_1024, uncertainty-aware truth table)

Primary rows: ensemble × focus parameters. Discordant diagnostics or CI boundary crossing → `INCONCLUSIVE`. Rank mean alone cannot certify SBC health.

In [5]:
reg = pd.read_csv(CSV / "residual_failure_registry_primary.csv")
display(reg[
    [
        "track",
        "parameter",
        "central_cov_90",
        "wilson_lo",
        "wilson_hi",
        "cov_state",
        "rank_mean",
        "rank_mean_ci_lo",
        "rank_mean_ci_hi",
        "rank_state",
        "ks_state",
        "residual_label_final",
        "matched48_material_delta",
    ]
])

,track,parameter,central_cov_90,wilson_lo,wilson_hi,cov_state,rank_mean,rank_mean_ci_lo,rank_mean_ci_hi,rank_state,ks_state,residual_label_final,matched48_material_delta
0,7d,mue,0.820312,0.795613,0.842617,unhealthy,0.518873,0.499571,0.538622,healthy,unhealthy,INCONCLUSIVE,True
1,7d,mui,0.981445,0.971202,0.988090,unhealthy,0.457001,0.444840,0.469430,crossing,unhealthy,INCONCLUSIVE,False
2,7d,tauA,0.823242,0.798682,0.845386,unhealthy,0.519168,0.500084,0.537900,healthy,unhealthy,INCONCLUSIVE,False
3,8d,mue,0.830078,0.805851,0.851838,crossing,0.519920,0.499219,0.539459,healthy,unhealthy,INCONCLUSIVE,True
4,8d,mui,0.979492,0.968852,0.986548,unhealthy,0.589398,0.578564,0.601520,unhealthy,unhealthy,RESIDUAL_FAILURE,False
5,8d,tauA,0.828125,0.803801,0.849996,unhealthy,0.524031,0.505913,0.543787,healthy,unhealthy,INCONCLUSIVE,False


## 5. Final Auth A decision

Auth B/B+/C were not executed. Sealed Bank was not generated, loaded, or evaluated.

In [6]:
display(Markdown(f"**Stopping decision:** `{gate['stopping_decision']}`"))
display(Markdown(f"**Reason:** {gate['reason']}"))
display(Markdown(
    f"Sealed opened: `{gate['sealed_bank_opened']}` · "
    f"Auth B/B+/C executed: `{gate['auth_b_executed']}` / "
    f"`{gate['auth_bplus_executed']}` / `{gate['auth_c_executed']}`"
))
display(Markdown(f"Auth C power note (future): {gate.get('auth_c_power_note', '')}"))

**Stopping decision:** `AUTH_A_COMPLETE_RESIDUAL_PRESENT — AUTH_B_NOT_EXECUTED`

**Reason:** residual_failure_count=1; Auth B not authorized in this run

Sealed opened: `False` · Auth B/B+/C executed: `False` / `False` / `False`

Auth C power note (future): n=200 Wilson CONFIRM probability at true 90% coverage ~35.2%; re-review sealed design before Auth C